# Graph Infrastructure Demo: US-201 & US-202

This notebook demonstrates the graph infrastructure built for the Vartakuni Vihāram TSP solver:

- **US-201**: Graph Builder Module - Building and analyzing metro network graphs
- **US-202**: Data Validation Pipeline - Validating metro network data integrity

## Singapore MRT/LRT Network

The Singapore Mass Rapid Transit (MRT) and Light Rail Transit (LRT) network comprises:
- **214 stations** (181 unique locations with interchange stations counted separately)
- **277 connections** (train lines + walking transfers)
- **15 lines** (8 MRT + 7 LRT lines)

In [ ]:
# Setup
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Imports
from src.graph import (
    MetroGraphBuilder,
    build_singapore_metro_graph,
    MetroDataValidator,
    validate_metro_data
)
import networkx as nx
import pandas as pd

# Data directory
data_dir = project_root / 'data' / 'raw'

print("✅ Setup complete")
print(f"📂 Data directory: {data_dir}")

## Part 1: US-202 - Data Validation Pipeline

Before building graphs, we validate the data to ensure quality and catch errors early.

### 1.1 Run Validation Pipeline

In [ ]:
# Run validation on Singapore data
print("Running data validation pipeline...\n")

report = validate_metro_data(data_dir)

# Print full report
report.print_report()

### 1.2 Validation Summary

The validation pipeline checks for:
- ✅ File existence
- ✅ Data integrity (required fields)
- ✅ Station ID references
- ✅ Travel time validity
- ✅ Distance validity
- ✅ Duplicate connections
- ✅ Graph connectivity

In [ ]:
# Detailed summary
summary = report.get_summary()

print("Validation Summary:")
print("=" * 50)
print(f"Total Issues: {sum(summary.values())}")
print(f"  - Errors:   {summary['error']}")
print(f"  - Warnings: {summary['warning']}")
print(f"  - Info:     {summary['info']}")
print()
print(f"Validation Status: {'✅ PASSED' if not report.has_errors() else '❌ FAILED'}")

# Show statistics
print("\nNetwork Statistics:")
print("=" * 50)
for key, value in report.stats.items():
    if isinstance(value, dict):
        print(f"{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

### 1.3 Custom Validation with MetroDataValidator

You can also use the `MetroDataValidator` class directly for more control:

In [ ]:
# Create validator instance
validator = MetroDataValidator()

# Run validation
custom_report = validator.validate(
    stations_csv=data_dir / 'stations.csv',
    connections_csv=data_dir / 'connections.csv',
    lines_csv=data_dir / 'lines.csv'
)

# Access specific validation results
print(f"Loaded {len(validator.stations)} stations")
print(f"Loaded {len(validator.connections)} connections")
print(f"Loaded {len(validator.lines)} lines")

# Show sample station data
print("\nSample Stations:")
for i, (station_id, data) in enumerate(list(validator.stations.items())[:5]):
    print(f"  {station_id}: {data['name']} (Line {data['line_code']})")

## Part 2: US-201 - Graph Builder Module

With validated data, we can now build the network graph for TSP algorithms.

### 2.1 Build Singapore Metro Graph (Quick Method)

In [ ]:
# Quick build using convenience function
graph = build_singapore_metro_graph(data_dir)

print("Graph Built Successfully!")
print("=" * 50)
print(f"Stations (nodes):     {graph.number_of_nodes()}")
print(f"Connections (edges):  {graph.number_of_edges()}")
print(f"Graph type:           {type(graph).__name__}")
print(f"Is connected:         {nx.is_connected(graph)}")

### 2.2 Build Graph with MetroGraphBuilder (Advanced)

In [ ]:
# Create builder instance
builder = MetroGraphBuilder()

# Build graph from CSV files
builder.build_graph(
    stations_csv=data_dir / 'stations.csv',
    connections_csv=data_dir / 'connections.csv',
    lines_csv=data_dir / 'lines.csv'
)

print("Graph built with MetroGraphBuilder")
print(f"Stations: {len(builder.stations)}")
print(f"Lines: {len(builder.lines)}")

### 2.3 Graph Statistics and Analysis

In [ ]:
# Get comprehensive graph statistics
stats = builder.get_graph_stats()

print("Detailed Graph Statistics:")
print("=" * 50)
for key, value in stats.items():
    print(f"{key}: {value}")

### 2.4 Validate Graph Connectivity

In [ ]:
# Validate connectivity
is_connected, components = builder.validate_connectivity()

print(f"Graph is connected: {is_connected}")
print(f"Number of components: {len(components)}")

if is_connected:
    print("\n✅ All stations are reachable from any starting point!")
else:
    print("\n⚠️ Graph has disconnected components:")
    for i, component in enumerate(components, 1):
        print(f"  Component {i}: {len(component)} stations")
        print(f"    Stations: {sorted(list(component)[:5])}")

### 2.5 Shortest Path Analysis

Find the shortest path between two stations using Dijkstra's algorithm.

In [ ]:
# Example 1: Jurong East to Changi Airport
start = 'NS1'  # Jurong East
end = 'CG2'    # Changi Airport

path, travel_time = builder.get_shortest_path(start, end)

print(f"Shortest Path: {builder.stations[start]['name']} → {builder.stations[end]['name']}")
print("=" * 70)
print(f"Route: {' → '.join(path)}")
print(f"Number of stations: {len(path)}")
print(f"Total travel time: {travel_time:.2f} minutes")

# Show station names
print("\nDetailed Route:")
for i, station_id in enumerate(path, 1):
    station_name = builder.stations[station_id]['name']
    line_code = builder.stations[station_id]['line_code']
    print(f"{i:2d}. [{line_code:3s}] {station_name}")

In [ ]:
# Example 2: Marina Bay to Sengkang
start = 'NS27'  # Marina Bay
end = 'NE16'    # Sengkang

path, travel_time = builder.get_shortest_path(start, end)

print(f"Shortest Path: {builder.stations[start]['name']} → {builder.stations[end]['name']}")
print("=" * 70)
print(f"Number of stations: {len(path)}")
print(f"Total travel time: {travel_time:.2f} minutes")

print("\nRoute:")
for i, station_id in enumerate(path, 1):
    station_name = builder.stations[station_id]['name']
    line_code = builder.stations[station_id]['line_code']
    print(f"{i:2d}. [{line_code:3s}] {station_name}")

### 2.6 Explore Station Data

Access detailed station and line information.

In [ ]:
# Get station information
station_id = 'CC1'  # Dhoby Ghaut
station_info = builder.get_station_info(station_id)

print(f"Station Information: {station_id}")
print("=" * 50)
for key, value in station_info.items():
    print(f"{key}: {value}")

In [ ]:
# Get line information
line_code = 'TE'  # Thomson-East Coast Line
line_info = builder.get_line_info(line_code)

print(f"Line Information: {line_code}")
print("=" * 50)
for key, value in line_info.items():
    print(f"{key}: {value}")

### 2.7 Analyze Interchange Stations

Find stations that serve multiple lines.

In [ ]:
# Find all interchange stations
from collections import defaultdict

# Group stations by name
stations_by_name = defaultdict(list)
for station_id, data in builder.stations.items():
    station_name = data['name']
    stations_by_name[station_name].append((station_id, data['line_code']))

# Find interchanges (stations with multiple codes)
interchange_stations = {
    name: stations 
    for name, stations in stations_by_name.items() 
    if len(stations) > 1
}

print(f"Found {len(interchange_stations)} interchange stations:\n")
for name, stations in sorted(interchange_stations.items())[:10]:
    lines = ', '.join([f"{code} ({id})" for id, code in sorted(stations)])
    print(f"  {name}: {lines}")

print(f"\n... and {len(interchange_stations) - 10} more interchange stations")

### 2.8 Network Degree Distribution

Analyze how many connections each station has.

In [ ]:
# Calculate degree for each node
degrees = dict(builder.graph.degree())

# Find stations with highest connectivity
top_connected = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:10]

print("Top 10 Most Connected Stations:")
print("=" * 70)
for station_id, degree in top_connected:
    station_name = builder.stations[station_id]['name']
    line_code = builder.stations[station_id]['line_code']
    print(f"{station_name:25s} [{line_code:3s}] - {degree} connections")

In [ ]:
# Degree distribution summary
from collections import Counter

degree_counts = Counter(degrees.values())

print("Degree Distribution:")
print("=" * 40)
for degree in sorted(degree_counts.keys()):
    count = degree_counts[degree]
    bar = '█' * (count // 5)
    print(f"{degree} connections: {count:3d} stations {bar}")

### 2.9 Direct Access to NetworkX Graph

The graph is a standard NetworkX graph, so you can use all NetworkX algorithms.

In [ ]:
# Access the raw NetworkX graph
G = builder.graph

# Example: Find graph diameter (longest shortest path)
diameter = nx.diameter(G)
print(f"Network diameter: {diameter} stations")
print("(This is the maximum number of stations in any shortest path)")

# Example: Calculate average clustering coefficient
avg_clustering = nx.average_clustering(G)
print(f"\nAverage clustering coefficient: {avg_clustering:.4f}")

# Example: Find center of the network (stations with minimum eccentricity)
center = nx.center(G)
print(f"\nNetwork center stations ({len(center)} stations):")
for station_id in center[:5]:
    station_name = builder.stations[station_id]['name']
    line_code = builder.stations[station_id]['line_code']
    print(f"  {station_name} [{line_code}]")

## Summary

This notebook demonstrated:

### US-202: Data Validation Pipeline
- ✅ Comprehensive validation of metro network data
- ✅ Detection of errors, warnings, and data quality issues
- ✅ Validation reports with detailed statistics
- ✅ Singapore MRT/LRT data passes all validation checks

### US-201: Graph Builder Module
- ✅ Building NetworkX graphs from CSV data
- ✅ Graph connectivity validation
- ✅ Shortest path finding (Dijkstra's algorithm)
- ✅ Station and line information access
- ✅ Network analysis and statistics
- ✅ Fully connected graph with 214 stations and 277 edges

### Next Steps
The validated, fully-connected graph is now ready for TSP algorithm implementation in **Epic 3: TSP Solver Development**!

---

*Vartakuni Vihāram (వర్తకుని విహారం) - A Seller's Journey*